# 07 — Story Building + Dot Detection

## What this notebook does

**Part 1 — Story Building**
Rebuilds Option A stories using parameters validated in notebook 06:
- Sliding window: 5-day windows, 2-day step
- Graph edges: BGE-M3 cosine similarity with `min_similarity=0.78`
- Story linking: majority vote + cross-gap merge with `T3=0.8`

**Part 2 — Dot Detection**
Tests and compares three methods for detecting dots (sub-event bursts within stories):
- **HDBSCAN** with adaptive time gap
- **Louvain** with adaptive time gap + exponential time decay
- **Connected Components** with 24h duration cap

Selected method: **Louvain with adaptive gap** — lowest singleton rates across
all story sizes, correct separation of temporally distinct developments,
and reasonable dot count for visualization.

## Result files
- `data/processed/stories.pkl` — `{story_id: [article_indices]}` — 18,473 stories
- `data/processed/dots_louvain.pkl` — `{story_id: [dot_dicts]}` — dots for 3,838 stories


In [27]:
from pathlib import Path
import pickle
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity as cos_sim_sk
from community import community_louvain
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_distances

REPO_ROOT = Path("../").resolve()
PARQUET_PATH = REPO_ROOT / "data" / "processed" / "articles_clean.parquet"
RANDOM_SEED = 42

# 1. Load Data

In [28]:
df = pd.read_parquet(PARQUET_PATH)
df["published_at_dt"] = pd.to_datetime(df["published_at_dt"], utc=True)

dates = (
    df["published_at_dt"].dt.tz_localize(None)
    if df["published_at_dt"].dt.tz is not None
    else df["published_at_dt"]
)
dates = dates.dt.normalize()

print(f"Articles: {len(df)}")

Articles: 30462


In [29]:
embs_all = np.load(REPO_ROOT / "data" / "processed" / "embeddings.npy")
emb_index_all = {i: embs_all[i] for i in range(len(df))}

print(f"Embeddings: {embs_all.shape}")

Embeddings: (30462, 1024)


# 2. Sliding Windows

## §3.2.2 — Sliding Window Construction

The NewsLens paper divides the timeline into overlapping windows of **N=5 days**, stepping forward by **2 days** at a time.
Each window is processed independently to find local topic clusters.
Overlapping windows ensure articles near a window boundary are seen in at least two windows,
which helps the story linking step (§3.2.3) connect events across time gaps.

Parameters used: `N_DAYS=5`, `STEP=2`

In [4]:
N_DAYS = 5
STEP = 2

unique_days = sorted(dates.unique())
windows = [
    (unique_days[i], unique_days[min(i + N_DAYS - 1, len(unique_days) - 1)])
    for i in range(0, len(unique_days) - N_DAYS + 1, STEP)
]
print(f"Windows: {len(windows)}")
for ws, we in windows:
    print(f"  {ws.strftime('%d %b')} → {we.strftime('%d %b')}")

Windows: 15
  20 Apr → 24 Apr
  22 Apr → 26 Apr
  24 Apr → 28 Apr
  26 Apr → 30 Apr
  28 Apr → 02 May
  30 Apr → 04 May
  02 May → 06 May
  04 May → 08 May
  06 May → 10 May
  08 May → 12 May
  10 May → 14 May
  12 May → 16 May
  14 May → 18 May
  16 May → 20 May
  18 May → 22 May


# 3. Embedding Graph + Louvain

## §3.2.2 — Graph Construction (Option A variant)

The paper builds a keyword co-occurrence graph per window using Jaccard similarity on TF-IDF keywords.
Option A keeps the exact same structure but replaces the edge weight:
instead of keyword Jaccard, edges are weighted by **BGE-M3 cosine similarity** between article embeddings.

An edge is added between two articles in the same window if their cosine similarity ≥ `min_similarity`.
Louvain community detection then partitions the graph into topic clusters.

**Selected parameter:** `min_similarity=0.78` — validated in notebook 06 as the best balance
between story quality (coherence=0.779, ratio=2.51x) and coverage comparable to the traditional baseline (38.5%).
Lower thresholds (≤0.70) allow the Bulgarian news semantic floor (~0.3–0.5) to connect unrelated articles into one massive cluster.

In [5]:
def build_topics_embed(
    df,
    emb_index_all,
    dates,
    windows,
    min_similarity: float = 0.78,
    random_seed: int = RANDOM_SEED,
):
    topics = []

    for ws, we in tqdm(windows, desc="Building embedding topics"):
        mask = (dates >= ws) & (dates <= we)
        window_indices = list(df[mask].index)

        if len(window_indices) < 2:
            topics.append({})
            continue

        embs = np.array([emb_index_all[i] for i in window_indices])
        sim_matrix = cos_sim_sk(embs)

        G = nx.Graph()
        G.add_nodes_from(range(len(window_indices)))

        for i in range(len(window_indices)):
            for j in range(i + 1, len(window_indices)):
                if sim_matrix[i, j] >= min_similarity:
                    G.add_edge(i, j, weight=float(sim_matrix[i, j]))

        partition = community_louvain.best_partition(G, random_state=random_seed)

        topic_map = {}
        for local_idx, topic_id in partition.items():
            topic_map[window_indices[local_idx]] = topic_id

        topics.append(topic_map)

    return topics

In [6]:
topics = build_topics_embed(df, emb_index_all, dates, windows, min_similarity=0.78)

Building embedding topics:   0%|          | 0/15 [00:00<?, ?it/s]

## §3.2.3 — Story Linking (Majority Vote + Cross-Gap Merge)

The paper links window-level topic clusters into cross-window stories using two rules:

1. **Majority vote** — for each cluster in a new window, look at how many of its articles
   were already assigned to an existing story. If more than half vote for the same story,
   that cluster is merged into it; otherwise it starts a new story.

2. **Cross-gap merge** — after all windows are processed, stories whose mean embedding
   similarity exceeds threshold `T3` and whose last-seen windows are adjacent are merged.
   This reconnects a story that briefly fell silent (no articles for one window step).

**Parameter:** `T3=0.8` — same as notebook 06 and the paper's recommended value.

In [7]:
def build_stories_embed(window_topics, emb_index_all, T3=0.8):
    article_story = {}
    story_alias = {}
    story_last_win = {}
    next_id = [0]

    def resolve(sid):
        while story_alias.get(sid, sid) != sid:
            sid = story_alias[sid]
        return sid

    def create_story(arts, win_idx):
        sid = next_id[0]
        next_id[0] += 1
        for a in arts:
            article_story[a] = sid
        story_last_win[sid] = win_idx

    def assign_to_story(arts, sid, win_idx):
        for a in arts:
            article_story[a] = sid
        story_last_win[sid] = win_idx

    def merge_stories(sid_keep, sid_drop):
        sid_keep = resolve(sid_keep)
        sid_drop = resolve(sid_drop)
        if sid_keep == sid_drop:
            return
        story_alias[sid_drop] = sid_keep

    for win_idx, partition in enumerate(tqdm(window_topics, desc="Linking")):
        clusters = defaultdict(list)
        for art_idx, cluster_id in partition.items():
            clusters[cluster_id].append(art_idx)
        for arts in clusters.values():
            prev = [resolve(article_story[a]) for a in arts if a in article_story]
            if not prev:
                create_story(arts, win_idx)
                continue
            vote = Counter(prev)
            top_sid, top_count = vote.most_common(1)[0]
            if top_count > len(arts) / 2:
                for other in list(vote.keys()):
                    if other != top_sid:
                        merge_stories(top_sid, other)
                assign_to_story(arts, top_sid, win_idx)
            else:
                create_story(arts, win_idx)

    # cross-gap merge using mean embeddings
    story_articles_tmp = defaultdict(list)
    for art_idx, sid in article_story.items():
        story_articles_tmp[resolve(sid)].append(art_idx)

    canonical_ids = sorted(story_articles_tmp.keys())
    mean_embs = np.array(
        [
            np.mean([emb_index_all[a] for a in story_articles_tmp[sid]], axis=0)
            for sid in canonical_ids
        ]
    )
    mean_embs = mean_embs / np.linalg.norm(mean_embs, axis=1, keepdims=True)
    sim_matrix = mean_embs @ mean_embs.T

    merged_count = 0
    for ri in range(len(canonical_ids)):
        for ci in range(ri + 1, len(canonical_ids)):
            if sim_matrix[ri, ci] >= T3:
                sid_a = resolve(canonical_ids[ri])
                sid_b = resolve(canonical_ids[ci])
                if sid_a == sid_b:
                    continue
                if abs(story_last_win.get(sid_a, -1) - story_last_win.get(sid_b, -1)) > 1:
                    merge_stories(sid_a, sid_b)
                    merged_count += 1

    story_articles = defaultdict(list)
    for art_idx, sid in article_story.items():
        story_articles[resolve(sid)].append(art_idx)
    story_articles = dict(story_articles)

    sizes = pd.Series({sid: len(arts) for sid, arts in story_articles.items()})
    print(f"T3={T3} | Stories: {len(story_articles):,} | Cross-gap merges: {merged_count}")
    print(f"Singletons: {(sizes==1).sum()} ({(sizes==1).mean():.1%}) | Largest: {sizes.max()}")
    return story_articles

In [8]:
stories = build_stories_embed(topics, emb_index_all, T3=0.8)

Linking:   0%|          | 0/15 [00:00<?, ?it/s]

T3=0.8 | Stories: 18,473 | Cross-gap merges: 1091
Singletons: 14635 (79.2%) | Largest: 236


## Save Stories

In [9]:
stories_path = REPO_ROOT / "data" / "processed" / "stories.pkl"
with open(stories_path, "wb") as f:
    pickle.dump(stories, f)

print(f"Saved {len(stories):,} stories → {stories_path}")

Saved 18,473 stories → /Users/ivanadonchevska/Projects/msc_thesis/data/processed/stories.pkl


In [30]:
with open(REPO_ROOT / "data" / "processed" / "stories.pkl", "rb") as f:
    stories = pickle.load(f)
print(f"Loaded {len(stories):,} stories")

Loaded 18,473 stories


# 4. Dot Detection

## §3.2.4 — Sub-Event Detection within Stories

A story groups all articles covering the same real-world event across its full lifetime.
A **dot** is a sub-cluster within a story: a burst of articles published within a short time window
about the same specific development (e.g. an initial report, a reaction, a follow-up).

Each dot becomes one point on the story timeline in the final visualisation.

## Methods Explored

Three approaches are implemented and compared in this notebook:

1. **HDBSCAN with adaptive gap** — precomputed distance matrix combining cosine distance and
   an adaptive time gap threshold (median inter-article gap × 2, floored at 1h, capped at 24h).
   Articles beyond the gap are disconnected; noise articles (label=-1) become singleton dots.

2. **Louvain with adaptive time decay** — cosine similarity graph where edge weights are
   attenuated by `exp(−time_gap / decay_hours)`. The decay constant is computed per story
   as the median inter-article gap (floored at 1h). Louvain community detection partitions
   the graph into dots.

3. **Connected Components (CC) with 24h duration cap** — Phase 1: connects all pairs with
   cosine similarity ≥ threshold into transitive components. Phase 2: recursively splits
   any component spanning >24h at its largest internal time gap.

All three methods are evaluated across five story-size buckets (2-5, 5-10, 10-20, 20-50, 50+ articles)
using singleton rate, intra-dot cohesion, and cohesion/separation ratio.

## Selected Method: Louvain with Adaptive Time Decay

Based on the comparison below, Louvain is selected as the final dot detection method.

**Graph construction:**
- Edge condition: cosine similarity ≥ `min_similarity` **and** time gap < `max_gap_cap_hours`
- Edge weight: `similarity × exp(−time_gap / decay_hours)` where
  `decay_hours = max(median_inter_article_gap, 1.0)` — adaptive per story
- Community detection: Louvain algorithm on the weighted graph

Parameters are explored with defaults (`min_similarity=0.6`, `max_gap_cap_hours=24h`) here
and tuned on manually labeled stories in notebook 12.

The `effective_start` timestamp used for dot ordering respects the `fetched_at` time for articles
where the published timestamp differs from the fetch time by more than 2.5 hours
(i.e. old articles re-indexed by the source).

In [31]:
def _effective_time(idx, df, update_threshold_hours=2.5):
    row = df.iloc[idx]
    published = row["published_at_dt"]
    fetched = pd.to_datetime(row["fetched_at"], utc=True)
    gap = abs((fetched - published).total_seconds()) / 3600
    return fetched if gap > update_threshold_hours else published

In [32]:
def detect_dots_louvain_embed(
    story_articles: list,
    df: pd.DataFrame,
    emb_index: dict,
    min_similarity: float = 0.6,
    max_gap_cap_hours: float = 24.0,
    random_seed: int = RANDOM_SEED,
) -> list:
    n = len(story_articles)
    times = [_effective_time(i, df) for i in story_articles]

    # adaptive decay: based on story's median inter-article gap
    sorted_times = sorted(times)
    if len(sorted_times) >= 2:
        gaps_h = [
            (sorted_times[i + 1] - sorted_times[i]).total_seconds() / 3600
            for i in range(len(sorted_times) - 1)
        ]
        decay_hours = max(float(np.median(gaps_h)), 1.0)
    else:
        decay_hours = 6.0

    embs = np.array([emb_index[i] for i in story_articles])
    sim_matrix = cos_sim_sk(embs)

    G = nx.Graph()
    G.add_nodes_from(range(n))
    for i in range(n):
        for j in range(i + 1, n):
            sim = float(sim_matrix[i, j])
            if sim < min_similarity:
                continue
            time_gap = abs((times[i] - times[j]).total_seconds()) / 3600
            if time_gap >= max_gap_cap_hours:
                continue
            weight = sim * np.exp(-time_gap / decay_hours)
            G.add_edge(i, j, weight=weight)

    if G.number_of_edges() == 0:
        dot_list = []
        for idx in story_articles:
            eff_t = _effective_time(idx, df)
            pub = df.iloc[idx]["published_at_dt"]
            dot_list.append(
                {
                    "indices": [idx],
                    "start": pub,
                    "end": pub,
                    "effective_start": eff_t,
                    "size": 1,
                    "sources": [df.iloc[idx]["source"]],
                }
            )
        return sorted(dot_list, key=lambda x: x["effective_start"])

    partition = community_louvain.best_partition(G, random_state=random_seed)
    dots = defaultdict(list)
    for local_idx, dot_id in partition.items():
        dots[dot_id].append(story_articles[local_idx])

    dot_list = []
    for indices in dots.values():
        dot_dates = [df.iloc[i]["published_at_dt"] for i in indices]
        eff_times = [_effective_time(i, df) for i in indices]
        dot_list.append(
            {
                "indices": indices,
                "start": min(dot_dates),
                "end": max(dot_dates),
                "effective_start": min(eff_times),
                "size": len(indices),
                "sources": df.iloc[indices]["source"].tolist(),
            }
        )
    dot_list.sort(key=lambda x: x["effective_start"])
    return dot_list

In [33]:
def detect_dots_hdbscan_embed(
    story_articles: list,
    df: pd.DataFrame,
    emb_index: dict,
    min_similarity: float = 0.6,
    max_gap_cap_hours: float = 24.0,
    min_cluster_size: int = 2,
) -> list:
    n = len(story_articles)

    if n == 1:
        idx = story_articles[0]
        pub = df.iloc[idx]["published_at_dt"]
        return [{
            "indices": [idx],
            "start": pub,
            "end": pub,
            "effective_start": _effective_time(idx, df),
            "size": 1,
            "sources": [df.iloc[idx]["source"]],
        }]

    times = [_effective_time(i, df) for i in story_articles]

    # adaptive gap: median of consecutive inter-article gaps * 2, floored at 1h
    sorted_times = sorted(times)
    if len(sorted_times) >= 2:
        gaps_h = [
            (sorted_times[i+1] - sorted_times[i]).total_seconds() / 3600
            for i in range(len(sorted_times) - 1)
        ]
        max_hours_gap = max(min(float(np.median(gaps_h)) * 2, max_gap_cap_hours), 1.0)
    else:
        max_hours_gap = max_gap_cap_hours

    embs = np.array([emb_index[i] for i in story_articles])
    sim_matrix = cos_sim_sk(embs)

    dist_matrix = np.full((n, n), 2.0)
    np.fill_diagonal(dist_matrix, 0.0)
    for i in range(n):
        for j in range(i + 1, n):
            time_gap = abs((times[i] - times[j]).total_seconds()) / 3600
            if time_gap >= max_hours_gap:
                continue
            sim = float(sim_matrix[i, j])
            if sim < min_similarity:
                continue
            dist_matrix[i, j] = 1.0 - sim
            dist_matrix[j, i] = 1.0 - sim

    labels = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=1,
        metric="precomputed",
        copy=False,
    ).fit_predict(dist_matrix)

    dots = defaultdict(list)
    next_id = int(max(labels)) + 1
    for local_idx, label in enumerate(labels):
        if label == -1:
            dots[next_id].append(story_articles[local_idx])
            next_id += 1
        else:
            dots[label].append(story_articles[local_idx])

    dot_list = []
    for indices in dots.values():
        dot_dates = [df.iloc[i]["published_at_dt"] for i in indices]
        eff_times = [_effective_time(i, df) for i in indices]
        dot_list.append({
            "indices": indices,
            "start": min(dot_dates),
            "end": max(dot_dates),
            "effective_start": min(eff_times),
            "size": len(indices),
            "sources": [df.iloc[i]["source"] for i in indices],
        })
    dot_list.sort(key=lambda x: x["effective_start"])
    return dot_list

In [34]:
def evaluate_dots(dots: list, emb_index: dict, verbose: bool = True) -> dict:
    if len(dots) < 2:
        return {}

    n_singletons = sum(1 for d in dots if d["size"] == 1)
    singleton_rate = n_singletons / len(dots)

    # silhouette score
    all_indices, labels = [], []
    for dot_id, dot in enumerate(dots):
        for idx in dot["indices"]:
            all_indices.append(idx)
            labels.append(dot_id)
    embs = np.array([emb_index[i] for i in all_indices])
    labels_arr = np.array(labels)
    if len(set(labels)) >= 2 and len(set(labels)) < len(all_indices):
        dist_matrix = cosine_distances(embs)
        sil = silhouette_score(dist_matrix, labels_arr, metric="precomputed")
    else:
        sil = None

    # intra-dot cohesion
    cohesion_scores = []
    for dot in dots:
        if len(dot["indices"]) < 2:
            continue
        e = np.array([emb_index[i] for i in dot["indices"]])
        sim = cos_sim_sk(e)
        n = len(dot["indices"])
        pairs = [sim[i, j] for i in range(n) for j in range(i + 1, n)]
        cohesion_scores.append(np.mean(pairs))

    # inter-dot separation
    separation_scores = []
    for i in range(len(dots)):
        for j in range(i + 1, len(dots)):
            e_i = np.array([emb_index[idx] for idx in dots[i]["indices"]])
            e_j = np.array([emb_index[idx] for idx in dots[j]["indices"]])
            separation_scores.append(float(np.mean(cos_sim_sk(e_i, e_j))))

    # temporal: mean dot duration in hours
    durations_h = [(d["end"] - d["start"]).total_seconds() / 3600 for d in dots]

    mean_coh = np.mean(cohesion_scores) if cohesion_scores else 0.0
    mean_sep = np.mean(separation_scores) if separation_scores else 0.0
    ratio = mean_coh / mean_sep if mean_sep > 0 else float("inf")
    mean_duration_h = np.mean(durations_h)
    pct_over_24h = np.mean([d > 24 for d in durations_h])

    if verbose:
        print(f"Total dots:          {len(dots)}")
        print(
            f"Singleton rate:      {singleton_rate:.1%}  ({n_singletons}/{len(dots)})"
        )
        print(
            f"Silhouette:          {sil:.4f}"
            if sil is not None
            else "Silhouette:          N/A"
        )
        print(f"Cohesion:            {mean_coh:.4f}")
        print(f"Separation:          {mean_sep:.4f}")
        print(f"Ratio (coh/sep):     {ratio:.2f}x")
        print(f"Mean dot duration:   {mean_duration_h:.1f}h")
        print(f"Dots spanning >24h:  {pct_over_24h:.1%}")

    return {
        "n_dots": len(dots),
        "singleton_rate": singleton_rate,
        "silhouette": sil,
        "cohesion": mean_coh,
        "separation": mean_sep,
        "ratio": ratio,
        "mean_duration_h": mean_duration_h,
        "pct_over_24h": pct_over_24h,
    }

## Run louvian_embed dot detection on all stories

In [35]:
all_dots_louvian = {}
for sid, arts in tqdm(stories.items(), desc="Detecting dots"):
    if not arts:
        continue
    emb_idx = {i: emb_index_all[i] for i in arts}
    all_dots_louvian[sid] = detect_dots_louvain_embed(arts, df, emb_idx)

total_dots = sum(len(dots) for dots in all_dots_louvian.values())
print(f"Stories processed: {len(all_dots_louvian):,}")
print(f"Total dots: {total_dots:,}")

Detecting dots:   0%|          | 0/18473 [00:00<?, ?it/s]

Stories processed: 18,473
Total dots: 20,974


In [36]:
rows_louvain = []
for sid, dots in all_dots_louvian.items():
    arts = stories[sid]
    emb_idx = {i: emb_index_all[i] for i in arts}
    r = evaluate_dots(dots, emb_idx, verbose=False)
    if r:
        rows_louvain.append({"sid": sid, "n_articles": len(arts), **r})

eval_louvain = pd.DataFrame(rows_louvain)

eval_louvain["size_bucket"] = pd.cut(
    eval_louvain["n_articles"],
    bins=[1, 5, 10, 20, 50, 300],
    labels=["2-5", "5-10", "10-20", "20-50", "50+"],
)

print(f"Stories evaluated: {len(eval_louvain):,}")
print()
print(
    eval_louvain.groupby("size_bucket", observed=True)[
        ["singleton_rate", "cohesion", "ratio", "mean_duration_h", "n_dots"]
    ]
    .mean()
    .round(3)
)
print()
print("Story counts per bucket:")
print(eval_louvain["size_bucket"].value_counts().sort_index())

Stories evaluated: 1,275

             singleton_rate  cohesion  ratio  mean_duration_h  n_dots
size_bucket                                                          
2-5                   0.623     0.464  0.572            1.894   2.133
5-10                  0.154     0.841  1.070            5.952   2.737
10-20                 0.209     0.817  1.099            7.432   5.000
20-50                 0.162     0.804  1.130            9.059   8.478
50+                   0.044     0.754  1.211            9.840  21.667

Story counts per bucket:
size_bucket
2-5      815
5-10     297
10-20    102
20-50     46
50+       15
Name: count, dtype: int64


In [37]:
def print_dots_for_story(sid):
    arts = stories[sid]
    dots = all_dots_louvian[sid]
    print(f"Story id={sid} | {len(arts)} articles → {len(dots)} dots")
    for i, dot in enumerate(dots):
        start = dot["effective_start"].strftime("%d %b %H:%M")
        end = dot["end"].strftime("%d %b %H:%M")
        duration_h = (dot["end"] - dot["start"]).total_seconds() / 3600
        sources = ", ".join(sorted(set(dot["sources"])))
        print(
            f"  Dot {i+1:2d} | {start} → {end} ({duration_h:.1f}h) | size={dot['size']} | {sources}"
        )
        for idx in dot["indices"][:2]:
            print(f"           - {df.iloc[idx]['title']}")
    print()

In [39]:
sid_2_5 = (
    eval_louvain[eval_louvain["size_bucket"] == "2-5"]
    .sample(1, random_state=42)["sid"]
    .iloc[0]
)
print_dots_for_story(sid_2_5)

Story id=3695 | 2 articles → 2 dots
  Dot  1 | 27 Apr 08:18 → 27 Apr 08:18 (0.0h) | size=1 | actualno
           - Гатев и Стоян се изправят един срещу друг в "Ергенът"
  Dot  2 | 28 Apr 09:31 → 28 Apr 09:31 (0.0h) | size=1 | fakti
           - Стоян и Гатев застават лице в лице в "Ергенът" тази седмица



In [100]:
def print_article_similarities(article_indices):
    embs = np.array([emb_index_all[i] for i in article_indices])
    sim_matrix = cos_sim_sk(embs)
    titles = [df.iloc[i]["title"] for i in article_indices]
    sim_df = pd.DataFrame(sim_matrix, index=titles, columns=titles)
    print(sim_df.round(3))

In [101]:
print_article_similarities(stories[3609])

                                                    Шофьор загина след удар на кола в трактор в Пловдивско  \
Шофьор загина след удар на кола в трактор в Пло...                                              1.000        
Човек загина в катастрофа между селата Стряма и...                                              0.772        
Кола се блъсна в трактор на пътя Стряма-Калеков...                                              0.811        
Млад шофьор загина при катастрофа с трактор в П...                                              0.930        

                                                    Човек загина в катастрофа между селата Стряма и Калековец край Пловдив  \
Шофьор загина след удар на кола в трактор в Пло...                                              0.772                        
Човек загина в катастрофа между селата Стряма и...                                              1.000                        
Кола се блъсна в трактор на пътя Стряма-Калеков...                     

In [135]:
sid_2_5 = (
    eval_louvain[eval_louvain["size_bucket"] == "5-10"]
    .sample(1)["sid"]
    .iloc[0]
)
print_dots_for_story(sid_2_5)

Story id=385 | 6 articles → 2 dots
  Dot  1 | 23 Apr 08:09 → 23 Apr 09:07 (1.0h) | size=4 | 24chasa, actualno, fakti, nova
           - Опасен товар: Хванаха стотици флакона ботулинов токсин в микробус
           - Задържаха 119 флакона ботулинов токсин на „Капитан Андреево“
  Dot  2 | 24 Apr 04:51 → 24 Apr 06:33 (1.7h) | size=2 | 24chasa, nova
           - Нелегален ботокс през границата: Какво показва случаят на „Капитан Андреево“
           - Хванатият на "Капитан Андреево" контрабанден ботулинов токсин е трябвало да бъде продаден във Великобритания



In [114]:
sid_2_5 = (
    eval_louvain[eval_louvain["size_bucket"] == "20-50"]
    .sample(1,)["sid"]
    .iloc[0]
)
print_dots_for_story(sid_2_5)

Story id=9660 | 39 articles → 11 dots
  Dot  1 | 21 Apr 02:48 → 21 Apr 15:32 (26.6h) | size=2 | 24chasa
           - Само три кораба са преминали през Ормузкия проток през последните 24 часа
           - Трафикът през Ормузкия проток остава практически в застой
  Dot  2 | 23 Apr 09:15 → 23 Apr 10:57 (1.7h) | size=4 | 24chasa, actualno, fakti, nova
           - Иран се похвали с първи събрани транзитни такси за Ормузкия проток
           - Иран обяви първи приходи от такси в Ормузкия проток на фона на напрежение и ограничен морски трафик
  Dot  3 | 24 Apr 15:28 → 24 Apr 15:48 (0.3h) | size=2 | 24chasa, fakti
           - Само 5 кораба са преминали през Ормузкия проток през последните 24 часа, сочат платформи за следене
           - Само пет кораба са преминали през Ормузкия проток за последните 24 часа
  Dot  4 | 27 Apr 13:40 → 27 Apr 13:40 (0.0h) | size=1 | 24chasa
           - Ирански депутат: Армията ни трябва да контролира Ормузкия проток
  Dot  5 | 05 May 18:28 → 05 May 18:28 (0.0h

## Run hdbscan_embed dot detection on all stories

In [40]:
all_dots_hdbscan = {}
for sid, arts in tqdm(stories.items(), desc="Detecting dots"):
    if not arts:
        continue
    emb_idx = {i: emb_index_all[i] for i in arts}
    all_dots_hdbscan[sid] = detect_dots_hdbscan_embed(arts, df, emb_idx)

total_dots = sum(len(dots) for dots in all_dots_hdbscan.values())
print(f"Stories processed: {len(all_dots_hdbscan):,}")
print(f"Total dots: {total_dots:,}")

Detecting dots:   0%|          | 0/18473 [00:00<?, ?it/s]

Stories processed: 18,473
Total dots: 25,885


In [41]:
rows_hdbscan = []
for sid, dots in all_dots_hdbscan.items():
    arts = stories[sid]
    emb_idx = {i: emb_index_all[i] for i in arts}
    r = evaluate_dots(dots, emb_idx, verbose=False)
    if r:
        rows_hdbscan.append({"sid": sid, "n_articles": len(arts), **r})

eval_hdbscan = pd.DataFrame(rows_hdbscan)

eval_hdbscan["size_bucket"] = pd.cut(
    eval_hdbscan["n_articles"],
    bins=[1, 5, 10, 20, 50, 300],
    labels=["2-5", "5-10", "10-20", "20-50", "50+"],
)

print(f"Stories evaluated: {len(eval_hdbscan):,}")
print()
print(
    eval_hdbscan.groupby("size_bucket", observed=True)[
        ["singleton_rate", "cohesion", "ratio", "mean_duration_h", "n_dots"]
    ]
    .mean()
    .round(3)
)
print()
print("Story counts per bucket:")
print(eval_hdbscan["size_bucket"].value_counts().sort_index())

Stories evaluated: 3,838

             singleton_rate  cohesion  ratio  mean_duration_h  n_dots
size_bucket                                                          
2-5                   0.952     0.043  0.054            0.521   2.489
5-10                  0.303     0.697  0.890           19.575   3.514
10-20                 0.261     0.824  1.100           24.119   5.216
20-50                 0.260     0.811  1.132           19.748  10.696
50+                   0.355     0.790  1.265            5.183  50.200

Story counts per bucket:
size_bucket
2-5      3356
5-10      319
10-20     102
20-50      46
50+        15
Name: count, dtype: int64


# Connected components

In [42]:
def detect_dots_connected_components(
    story_articles: list,
    df: pd.DataFrame,
    emb_index: dict,
    min_similarity: float = 0.7,
    max_dot_duration_hours: float = 24.0,
) -> list:
    n = len(story_articles)
    if n == 1:
        idx = story_articles[0]
        pub = df.iloc[idx]["published_at_dt"]
        return [{
            "indices": [idx],
            "start": pub,
            "end": pub,
            "effective_start": _effective_time(idx, df),
            "size": 1,
            "sources": [df.iloc[idx]["source"]],
        }]
    
    times = [_effective_time(i, df) for i in story_articles]
    embs = np.array([emb_index[i] for i in story_articles])
    sim_matrix = cos_sim_sk(embs)

    # Phase 1: connected components by similarity only
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for i in range(n):
        for j in range(i + 1, n):
            if sim_matrix[i, j] >= min_similarity:
                G.add_edge(i, j)

    def make_dot(local_indices):
        art_indices = [story_articles[i] for i in local_indices]
        dot_dates = [df.iloc[a]["published_at_dt"] for a in art_indices]
        eff_times = [_effective_time(a, df) for a in art_indices]
        return {
            "indices": art_indices,
            "start": min(dot_dates),
            "end": max(dot_dates),
            "effective_start": min(eff_times),
            "size": len(art_indices),
            "sources": [df.iloc[a]["source"] for a in art_indices],
        }

    def split_component(local_indices):
        if not local_indices:
            return []
        sorted_idx = sorted(local_indices, key=lambda i: times[i])
        comp_times = [times[i] for i in sorted_idx]
        duration_h = (comp_times[-1] - comp_times[0]).total_seconds() / 3600

        if duration_h <= max_dot_duration_hours or len(sorted_idx) == 1:
            return [make_dot(sorted_idx)]

        gaps = [
            (comp_times[k + 1] - comp_times[k]).total_seconds() / 3600
            for k in range(len(comp_times) - 1)
        ]
        split_at = int(np.argmax(gaps)) + 1
        return split_component(sorted_idx[:split_at]) + split_component(
            sorted_idx[split_at:]
        )

    dot_list = []
    for component in nx.connected_components(G):
        dot_list.extend(split_component(list(component)))

    dot_list.sort(key=lambda x: x["effective_start"])
    return dot_list

In [43]:
arts = stories[9660]
emb_idx = {i: emb_index_all[i] for i in arts}
dots = detect_dots_connected_components(arts, df, emb_idx)
for i, d in enumerate(dots):
    duration_h = (d["end"] - d["start"]).total_seconds() / 3600
    sources = ", ".join(sorted(set(d["sources"])))
    print(
        f"Dot {i+1:2d} | {d['effective_start'].strftime('%d %b %H:%M')} → {d['end'].strftime('%d %b %H:%M')} ({duration_h:.1f}h) | size={d['size']} | {sources}"
    )
    for idx in d["indices"]:
        print(f"  - {df.iloc[idx]['title']}")
    print()

Dot  1 | 21 Apr 02:48 → 21 Apr 15:32 (26.6h) | size=2 | 24chasa
  - Трафикът през Ормузкия проток остава практически в застой
  - Само три кораба са преминали през Ормузкия проток през последните 24 часа

Dot  2 | 23 Apr 09:15 → 23 Apr 10:57 (1.7h) | size=4 | 24chasa, actualno, fakti, nova
  - Иран: Получихме първите си приходи от транзитни такси в Ормузкия проток
  - Иран отчита първи приходи от такси в Ормузкия проток
  - Иран се похвали с първи събрани транзитни такси за Ормузкия проток
  - Иран обяви първи приходи от такси в Ормузкия проток на фона на напрежение и ограничен морски трафик

Dot  3 | 24 Apr 15:28 → 24 Apr 15:48 (0.3h) | size=2 | 24chasa, fakti
  - Само 5 кораба са преминали през Ормузкия проток през последните 24 часа, сочат платформи за следене
  - Само пет кораба са преминали през Ормузкия проток за последните 24 часа

Dot  4 | 27 Apr 13:40 → 27 Apr 13:40 (0.0h) | size=1 | 24chasa
  - Ирански депутат: Армията ни трябва да контролира Ормузкия проток

Dot  5 | 05 May 

# Apply on all stories

In [44]:
all_dots_cc = {}
for sid, arts in tqdm(stories.items(), desc="Detecting dots"):
    if not arts:
        continue
    emb_idx = {i: emb_index_all[i] for i in arts}
    all_dots_cc[sid] = detect_dots_connected_components(arts, df, emb_idx)

total_dots = sum(len(dots) for dots in all_dots_cc.values())
print(f"Stories processed: {len(all_dots_cc):,}")
print(f"Total dots: {total_dots:,}")

Detecting dots:   0%|          | 0/18473 [00:00<?, ?it/s]

Stories processed: 18,473
Total dots: 20,652


In [45]:
rows_cc = []
for sid, dots in all_dots_cc.items():
    arts = stories[sid]
    emb_idx = {i: emb_index_all[i] for i in arts}
    r = evaluate_dots(dots, emb_idx, verbose=False)
    if r:
        rows_cc.append({"sid": sid, "n_articles": len(arts), **r})

eval_cc = pd.DataFrame(rows_cc)
eval_cc["size_bucket"] = pd.cut(
    eval_cc["n_articles"],
    bins=[1, 5, 10, 20, 50, 300],
    labels=["2-5", "5-10", "10-20", "20-50", "50+"],
    include_lowest=True,
)

print(f"Stories evaluated: {len(eval_cc):,}")
print()
print(
    eval_cc.groupby("size_bucket", observed=True)[
        ["singleton_rate", "cohesion", "ratio", "mean_duration_h", "n_dots"]
    ]
    .mean()
    .round(3)
)
print()
print("Story counts per bucket:")
print(eval_cc["size_bucket"].value_counts().sort_index())

Stories evaluated: 1,101

             singleton_rate  cohesion  ratio  mean_duration_h  n_dots
size_bucket                                                          
2-5                   0.719     0.423  0.526            1.779   2.153
5-10                  0.310     0.841  1.095            6.212   2.812
10-20                 0.271     0.821  1.108            8.063   4.650
20-50                 0.243     0.805  1.143            7.838   8.478
50+                   0.118     0.724  1.154            9.847  17.667

Story counts per bucket:
size_bucket
2-5      733
5-10     207
10-20    100
20-50     46
50+       15
Name: count, dtype: int64


# Summary

In [46]:
metrics = ["singleton_rate", "cohesion", "ratio", "mean_duration_h", "n_dots"]
buckets = ["2-5", "5-10", "10-20", "20-50", "50+"]

for bucket in buckets:
    print(f"\n{'='*50}")
    print(f"Bucket: {bucket}")
    print(f"{'='*50}")
    rows = []
    for name, ev in [
        ("Louvain", eval_louvain),
        ("HDBSCAN", eval_hdbscan),
        ("CC", eval_cc),
    ]:
        subset = ev[ev["size_bucket"] == bucket]
        if len(subset) == 0:
            continue
        row = subset[metrics].mean().round(3).to_dict()
        row["method"] = name
        row["n_stories"] = len(subset)
        rows.append(row)
    print(pd.DataFrame(rows).set_index("method")[["n_stories"] + metrics])


Bucket: 2-5
         n_stories  singleton_rate  cohesion  ratio  mean_duration_h  n_dots
method                                                                      
Louvain        815           0.623     0.464  0.572            1.894   2.133
HDBSCAN       3356           0.952     0.043  0.054            0.521   2.489
CC             733           0.719     0.423  0.526            1.779   2.153

Bucket: 5-10
         n_stories  singleton_rate  cohesion  ratio  mean_duration_h  n_dots
method                                                                      
Louvain        297           0.154     0.841  1.070            5.952   2.737
HDBSCAN        319           0.303     0.697  0.890           19.575   3.514
CC             207           0.310     0.841  1.095            6.212   2.812

Bucket: 10-20
         n_stories  singleton_rate  cohesion  ratio  mean_duration_h  n_dots
method                                                                      
Louvain        102           0.209

# Comparation louvian vs cc

In [144]:
arts = stories[2817]
emb_idx = {i: emb_index_all[i] for i in arts}
dots_lv = detect_dots_louvain_embed(arts, df, emb_idx)
for i, d in enumerate(dots_lv):
    duration_h = (d["end"] - d["start"]).total_seconds() / 3600
    sources = ", ".join(sorted(set(d["sources"])))
    print(
        f"Dot {i+1:2d} | {d['effective_start'].strftime('%d %b %H:%M')} → {d['end'].strftime('%d %b %H:%M')} ({duration_h:.1f}h) | size={d['size']} | {sources}"
    )
    for idx in d["indices"]:
        print(f"  - {df.iloc[idx]['title']}")
    print()

Dot  1 | 21 Apr 10:26 → 22 Apr 04:26 (18.8h) | size=3 | 24chasa, fakti
  - Взривове в Одеса при атака с дронове
  - Най-малко 25 души са ранени при руски удари по Украйна тази нощ, заяви Киев
  - Десетки ранени при руски удари с дронове в Украйна

Dot  2 | 22 Apr 05:01 → 22 Apr 06:25 (1.4h) | size=3 | 24chasa, fakti
  - Русия заяви, че тази нощ е свалила 155 украински дрона
  - Украйна е неутрализирала 189 от 215 руски дрона тази нощ
  - Русия съобщи за свалени 155 украински дрона за една нощ

Dot  3 | 23 Apr 04:18 → 23 Apr 10:19 (6.0h) | size=6 | 24chasa, fakti, nova
  - Трима души загинаха и 10 бяха ранени при руска атака с дрон срещу град Днипро
  - Двама са загинали и осем са ранени при руска атака в Днепър
  - Жертви и ранени при удар по Днипро, засилени атаки в Харковска област
  - Русия нанесе удари в Запорожка, Житомирска област и Днипро, има загинали
  - Русия нанесе удари в Запорожка и Житомирска област, двама са загинали
  - Един е убит, а няколко са ранени при украинска ата

In [143]:
arts = stories[2817]
emb_idx = {i: emb_index_all[i] for i in arts}
dots_cc = detect_dots_connected_components(arts, df, emb_idx)
for i, d in enumerate(dots_cc):
    duration_h = (d["end"] - d["start"]).total_seconds() / 3600
    sources = ", ".join(sorted(set(d["sources"])))
    print(
        f"Dot {i+1:2d} | {d['effective_start'].strftime('%d %b %H:%M')} → {d['end'].strftime('%d %b %H:%M')} ({duration_h:.1f}h) | size={d['size']} | {sources}"
    )
    for idx in d["indices"]:
        print(f"  - {df.iloc[idx]['title']}")
    print()

Dot  1 | 21 Apr 10:26 → 22 Apr 06:25 (20.8h) | size=6 | 24chasa, fakti
  - Десетки ранени при руски удари с дронове в Украйна
  - Най-малко 25 души са ранени при руски удари по Украйна тази нощ, заяви Киев
  - Взривове в Одеса при атака с дронове
  - Русия съобщи за свалени 155 украински дрона за една нощ
  - Украйна е неутрализирала 189 от 215 руски дрона тази нощ
  - Русия заяви, че тази нощ е свалила 155 украински дрона

Dot  2 | 23 Apr 04:18 → 23 Apr 10:19 (6.0h) | size=6 | 24chasa, fakti, nova
  - Жертви и ранени при удар по Днипро, засилени атаки в Харковска област
  - Двама са загинали и осем са ранени при руска атака в Днепър
  - Един е убит, а няколко са ранени при украинска атака с дрон срещу Новокуйбишевск
  - Русия нанесе удари в Запорожка и Житомирска област, двама са загинали
  - Трима души загинаха и 10 бяха ранени при руска атака с дрон срещу град Днипро
  - Русия нанесе удари в Запорожка, Житомирска област и Днипро, има загинали

Dot  3 | 24 Apr 04:07 → 24 Apr 09:35 (5

In [138]:
diff_stories = []
for sid in all_dots_louvian:
    if sid not in all_dots_cc:
        continue
    n_lv = len(all_dots_louvian[sid])
    n_cc = len(all_dots_cc[sid])
    if n_lv != n_cc:
        diff_stories.append(
            {
                "sid": sid,
                "n_articles": len(stories[sid]),
                "n_dots_louvain": n_lv,
                "n_dots_cc": n_cc,
                "diff": n_lv - n_cc,
            }
        )

diff_df = pd.DataFrame(diff_stories).sort_values("n_articles", ascending=False)
print(f"Stories where dot count differs: {len(diff_df)}")
print()
print(diff_df.head(20))

Stories where dot count differs: 447

       sid  n_articles  n_dots_louvain  n_dots_cc  diff
6    12337         236              31         35    -4
29    2817         234              46         30    16
25   12030         234              39         21    18
0    17045         221              42         35     7
79    9257         158              15         13     2
156  15389         157              17         12     5
239  13417         103              12         10     2
84    7330          96              16         12     4
103  10643          80              16         10     6
20    3077          79              16         20    -4
83   16792          78              16         18    -2
42    7842          70              17         16     1
114  13438          65              15         11     4
150  10122          65              15         11     4
159  11769          51              12         11     1
8     1831          49              12         10     2
189   8768

# Dot Detection — Method Comparison

## Methods Evaluated

Three approaches were tested for detecting dots (sub-event bursts) within stories:

1. **HDBSCAN with adaptive gap** — precomputed distance matrix combining cosine distance
   and an adaptive time gap threshold (median inter-article gap × 2, floored at 1h,
   capped at 24h). Articles beyond the gap are disconnected. Noise articles (label=-1)
   become singleton dots.

2. **Louvain with adaptive gap** — cosine similarity graph with exponential time decay
   on edge weights. The decay constant is computed per story from the median
   inter-article gap. Louvain community detection partitions the graph into dots.

3. **Connected Components (CC)** — Phase 1: connects all articles with sim ≥ 0.7 into
   connected components (transitive). Phase 2: recursively splits components spanning
   >24h at their largest time gap.

All three methods use the same adaptive time gap derived from each story's own
publishing rhythm, making the comparison fair.

## Results by Story Size

| Bucket | Method | Singleton rate | Cohesion | Duration (h) | n_dots |
|---|---|---|---|---|---|
| 2-5 | **Louvain** | **0.623** | **0.464** | 1.9h | 2.1 |
| | HDBSCAN | 0.952 | 0.043 | 0.5h | 2.5 |
| | CC | 0.719 | 0.423 | 1.8h | 2.2 |
| 5-10 | **Louvain** | **0.154** | **0.841** | 6.0h | 2.7 |
| | HDBSCAN | 0.303 | 0.697 | 19.6h | 3.5 |
| | CC | 0.310 | 0.841 | 6.2h | 2.8 |
| 10-20 | **Louvain** | **0.209** | 0.817 | 7.4h | 5.0 |
| | HDBSCAN | 0.261 | **0.824** | 24.1h | 5.2 |
| | CC | 0.271 | 0.821 | 8.1h | 4.7 |
| 20-50 | **Louvain** | **0.162** | 0.804 | 9.1h | 8.5 |
| | HDBSCAN | 0.260 | **0.811** | 19.7h | 10.7 |
| | CC | 0.243 | 0.805 | 7.8h | 8.5 |
| 50+ | **Louvain** | **0.044** | 0.754 | 9.8h | 21.7 |
| | HDBSCAN | 0.355 | **0.790** | 5.2h | 50.2 |
| | CC | 0.118 | 0.724 | 9.8h | 17.7 |

## Key Findings

**HDBSCAN** has two structural problems that persist even with adaptive gap tuning:
- **Chain effect**: articles connect transitively through intermediate neighbours,
  producing clusters that span 19-24h even when individual gaps are small
- **Noise labeling**: any article that doesn't fit a dense cluster becomes a singleton,
  causing 95% singleton rate on small stories and 35% on large ones
- **50+ stories**: produces ~50 dots per story — too many for a timeline visualization

**Louvain and CC** produce very similar results on most stories. Quantitatively,
Louvain has lower singleton rates across all buckets. Qualitatively, CC's transitive
connectivity causes over-merging on complex stories: distinct developments within
24h (e.g. a ceasefire proposal and a drone attack) land in one dot because both
articles connect to shared "topic" intermediates.

## Selected Method: Louvain with Adaptive Gap

Louvain is selected as the final dot detection method because:
- Lowest singleton rates across all story sizes (4% for 50+ stories)
- Correctly separates temporally distinct developments within the same story
- Reasonable dot count for visualization (~22 dots for large stories)
- The adaptive time decay handles both fast-breaking and slow daily stories
  without manual parameter tuning

In [24]:
dots_path = REPO_ROOT / "data" / "processed" / "dots_louvain.pkl"
with open(dots_path, "wb") as f:
    pickle.dump(all_dots_louvian, f)

print(f"Saved dots for {len(all_dots_louvian):,} stories → {dots_path}")

Saved dots for 18,473 stories → /Users/ivanadonchevska/Projects/msc_thesis/data/processed/dots_louvain.pkl


<table style="background-color:#f0f0f0; border-collapse:collapse; width:100%; color:#111">
  <thead>
    <tr style="background-color:#bdbdbd; color:#111">
      <th style="padding:8px; border:1px solid #999">Bucket</th>
      <th style="padding:8px; border:1px solid #999">Method</th>
      <th style="padding:8px; border:1px solid #999">Singleton rate</th>
      <th style="padding:8px; border:1px solid #999">Cohesion</th>
      <th style="padding:8px; border:1px solid #999">Mean duration (h)</th>
      <th style="padding:8px; border:1px solid #999">n_dots</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding:8px; border:1px solid #999" rowspan="3"><b>2-5</b></td>
      <td style="padding:8px; border:1px solid #999; font-weight:bold">Louvain *</td><td style="padding:8px; border:1px solid #999"><b>0.623</b></td><td style="padding:8px; border:1px solid #999"><b>0.464</b></td><td style="padding:8px; border:1px solid #999">1.9</td><td style="padding:8px; border:1px solid #999">2.1</td></tr>
    <tr><td style="padding:8px; border:1px solid #999">HDBSCAN</td><td style="padding:8px; border:1px solid #999">0.952</td><td style="padding:8px; border:1px solid #999">0.043</td><td style="padding:8px; border:1px solid #999">0.5</td><td style="padding:8px; border:1px solid #999">2.5</td></tr>
    <tr><td style="padding:8px; border:1px solid #999">CC</td><td style="padding:8px; border:1px solid #999">0.719</td><td style="padding:8px; border:1px solid #999">0.423</td><td style="padding:8px; border:1px solid #999">1.8</td><td style="padding:8px; border:1px solid #999">2.2</td></tr>
    <tr style="background-color:#ebebeb"><td style="padding:8px; border:1px solid #999" rowspan="3"><b>5-10</b></td>
      <td style="padding:8px; border:1px solid #999"><b>Louvain *</b></td><td style="padding:8px; border:1px solid #999"><b>0.154</b></td><td style="padding:8px; border:1px solid #999"><b>0.841</b></td><td style="padding:8px; border:1px solid #999">6.0</td><td style="padding:8px; border:1px solid #999">2.7</td></tr>
    <tr style="background-color:#ebebeb"><td style="padding:8px; border:1px solid #999">HDBSCAN</td><td style="padding:8px; border:1px solid #999">0.303</td><td style="padding:8px; border:1px solid #999">0.697</td><td style="padding:8px; border:1px solid #999">19.6</td><td style="padding:8px; border:1px solid #999">3.5</td></tr>
    <tr style="background-color:#ebebeb"><td style="padding:8px; border:1px solid #999">CC</td><td style="padding:8px; border:1px solid #999">0.310</td><td style="padding:8px; border:1px solid #999">0.841</td><td style="padding:8px; border:1px solid #999">6.2</td><td style="padding:8px; border:1px solid #999">2.8</td></tr>
    <tr><td style="padding:8px; border:1px solid #999" rowspan="3"><b>10-20</b></td>
      <td style="padding:8px; border:1px solid #999"><b>Louvain *</b></td><td style="padding:8px; border:1px solid #999"><b>0.209</b></td><td style="padding:8px; border:1px solid #999">0.817</td><td style="padding:8px; border:1px solid #999">7.4</td><td style="padding:8px; border:1px solid #999">5.0</td></tr>
    <tr><td style="padding:8px; border:1px solid #999">HDBSCAN</td><td style="padding:8px; border:1px solid #999">0.261</td><td style="padding:8px; border:1px solid #999"><b>0.824</b></td><td style="padding:8px; border:1px solid #999">24.1</td><td style="padding:8px; border:1px solid #999">5.2</td></tr>
    <tr><td style="padding:8px; border:1px solid #999">CC</td><td style="padding:8px; border:1px solid #999">0.271</td><td style="padding:8px; border:1px solid #999">0.821</td><td style="padding:8px; border:1px solid #999">8.1</td><td style="padding:8px; border:1px solid #999">4.7</td></tr>
    <tr style="background-color:#ebebeb"><td style="padding:8px; border:1px solid #999" rowspan="3"><b>20-50</b></td>
      <td style="padding:8px; border:1px solid #999"><b>Louvain *</b></td><td style="padding:8px; border:1px solid #999"><b>0.162</b></td><td style="padding:8px; border:1px solid #999">0.804</td><td style="padding:8px; border:1px solid #999">9.1</td><td style="padding:8px; border:1px solid #999">8.5</td></tr>
    <tr style="background-color:#ebebeb"><td style="padding:8px; border:1px solid #999">HDBSCAN</td><td style="padding:8px; border:1px solid #999">0.260</td><td style="padding:8px; border:1px solid #999"><b>0.811</b></td><td style="padding:8px; border:1px solid #999">19.7</td><td style="padding:8px; border:1px solid #999">10.7</td></tr>
    <tr style="background-color:#ebebeb"><td style="padding:8px; border:1px solid #999">CC</td><td style="padding:8px; border:1px solid #999">0.243</td><td style="padding:8px; border:1px solid #999">0.805</td><td style="padding:8px; border:1px solid #999">7.8</td><td style="padding:8px; border:1px solid #999">8.5</td></tr>
    <tr><td style="padding:8px; border:1px solid #999" rowspan="3"><b>50+</b></td>
      <td style="padding:8px; border:1px solid #999"><b>Louvain *</b></td><td style="padding:8px; border:1px solid #999"><b>0.044</b></td><td style="padding:8px; border:1px solid #999">0.754</td><td style="padding:8px; border:1px solid #999">9.8</td><td style="padding:8px; border:1px solid #999">21.7</td></tr>
    <tr><td style="padding:8px; border:1px solid #999">HDBSCAN</td><td style="padding:8px; border:1px solid #999">0.355</td><td style="padding:8px; border:1px solid #999"><b>0.790</b></td><td style="padding:8px; border:1px solid #999">5.2</td><td style="padding:8px; border:1px solid #999">50.2</td></tr>
    <tr><td style="padding:8px; border:1px solid #999">CC</td><td style="padding:8px; border:1px solid #999">0.118</td><td style="padding:8px; border:1px solid #999">0.724</td><td style="padding:8px; border:1px solid #999">9.8</td><td style="padding:8px; border:1px solid #999">17.7</td></tr>
  </tbody>
</table>
<p style="color:#111">* Selected method | Bold values = best per bucket per metric</p>

| | Count | % of total |
|---|---|---|
| Total stories | 18,473 | 100% |
| Single-article stories | 14,635 | 79.2% |
| Multi-article stories (2+ articles) | 3,838 | 20.8% |

===
| | Louvain | HDBSCAN | CC |
|---|---|---|---|
| Total dots produced | 20,974 | 25,885 | 20,652 |
| Multi-article stories → 2+ dots (evaluated) | 1,275 (33.2%) | 3,838 (100%) | 1,101 (28.7%) |
| Multi-article stories → collapsed to 1 dot | 2,563 (66.8%) | 0 (0%) | 2,737 (71.3%) |

Note: HDBSCAN's 100% evaluation rate is not a quality signal — it results from its noise-labeling behavior, which forces almost every article into its own singleton dot, artificially producing 2+ dots per story regardless of semantic coherence (cohesion=0.043 in the 2-5 bucket).

<table style="background-color:#f0f0f0; border-collapse:collapse; width:100%; color:#111">
  <thead>
    <tr style="background-color:#bdbdbd; color:#111">
      <th style="padding:8px; border:1px solid #999" colspan="3">Dataset Overview</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding:8px; border:1px solid #999">Total stories</td><td style="padding:8px; border:1px solid #999">18,473</td><td style="padding:8px; border:1px solid #999">100%</td></tr>
    <tr style="background-color:#ebebeb"><td style="padding:8px; border:1px solid #999">Single-article stories </td><td style="padding:8px; border:1px solid #999">14,635</td><td style="padding:8px; border:1px solid #999">79.2%</td></tr>
    <tr><td style="padding:8px; border:1px solid #999">Multi-article stories (2+ articles)</td><td style="padding:8px; border:1px solid #999">3,838</td><td style="padding:8px; border:1px solid #999">20.8%</td></tr>
  </tbody>
</table>

<br>

<table style="background-color:#f0f0f0; border-collapse:collapse; width:100%; color:#111">
  <thead>
    <tr style="background-color:#bdbdbd; color:#111">
      <th style="padding:8px; border:1px solid #999">Metric</th>
      <th style="padding:8px; border:1px solid #999">Louvain</th>
      <th style="padding:8px; border:1px solid #999">HDBSCAN</th>
      <th style="padding:8px; border:1px solid #999">CC</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding:8px; border:1px solid #999">Total dots produced</td><td style="padding:8px; border:1px solid #999">20,974</td><td style="padding:8px; border:1px solid #999">25,885</td><td style="padding:8px; border:1px solid #999">20,652</td></tr>
    <tr style="background-color:#ebebeb"><td style="padding:8px; border:1px solid #999">Multi-article stories → 2+ dots (evaluated)</td><td style="padding:8px; border:1px solid #999">1,275 (33.2%)</td><td style="padding:8px; border:1px solid #999">3,838 (100%)</td><td style="padding:8px; border:1px solid #999">1,101 (28.7%)</td></tr>
    <tr><td style="padding:8px; border:1px solid #999">Multi-article stories → collapsed to 1 dot</td><td style="padding:8px; border:1px solid #999">2,563 (66.8%)</td><td style="padding:8px; border:1px solid #999">0 (0%)</td><td style="padding:8px; border:1px solid #999">2,737 (71.3%)</td></tr>
  </tbody>
</table>
<p style="color:#111"></p>
